# MEMC cvičení 3 – Diskrétní rozdělení

---

Dosud jsme brali generátory hotové (`random.random()`, `random.gauss()`). Teď se naučíme **vyrábět vlastní**. Začneme diskrétními rozděleními, protože jsou nejjednodušší a jejich myšlenka se vrací všude:

> Z jednoho rovnoměrného čísla $\gamma \in [0, 1)$ vyrobíme libovolnou náhodnou veličinu tak, že $[0, 1)$ rozdělíme na kousky a podíváme se, do kterého $\gamma$ padlo.

V sešitu uvidíte:

- **základní (lineární) algoritmus** a jeho složitost
- jak ho zrychlit **přeuspořádáním** pravděpodobností
- **půlení intervalu** (bisekci) nad kumulativními součty
- **Poissonovo rozdělení** a pěknou chybu, která se do něj snadno vloudí
- geometrické rozdělení (potkali jsme ho ve cvičení 2)

In [ ]:
import math
import random
import time
from bisect import bisect_right
from itertools import accumulate

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def estimate(sampler, n):
    '''Běžné Monte Carlo: průměr n volání generátoru a jeho směrodatná chyba sigma/sqrt(n).'''
    s = s2 = 0.0
    for _ in range(n):
        x = sampler()
        s += x
        s2 += x * x
    mean = s / n
    var = max(s2 / n - mean * mean, 0.0)
    return mean, math.sqrt(var / n)


def check(name, mean, se, exact, k=5):
    '''Vypíše odhad a ověří, že leží do k směrodatných chyb od přesné hodnoty.'''
    print(f"{name}: {mean:.5f} +- {se:.5f}  (přesně {exact:.5f})")
    assert abs(mean - exact) <= k * se, name

## 1. Základní algoritmus

Máme diskrétní veličinu s hodnotami $0, 1, \dots, n-1$ a pravděpodobnostmi $p_0, \dots, p_{n-1}$ (součet $1$). Interval $[0, 1)$ rozdělíme na úseky délek $p_0, p_1, \dots$:

$$[0, p_0),\ [p_0, p_0 + p_1),\ [p_0 + p_1, p_0 + p_1 + p_2),\ \dots$$

Rovnoměrné $\gamma$ padne do úseku $k$ s pravděpodobností rovnou jeho délce, tedy přesně $p_k$. Výsledkem je index $k$ úseku, do kterého $\gamma$ padlo.

**Algoritmus:** $S \leftarrow \gamma$; pro $k = 0, 1, \dots$ odečti $S \leftarrow S - p_k$; jakmile je $S < 0$, vrať $k$.

Je to **inverzní transformace** v diskrétním případě: hledáme nejmenší $k$ s $\gamma < F(k) = p_0 + \dots + p_k$.

Obrázek algoritmu: vlevo je interval $[0, 1)$ rozdělený na úseky o délkách $p_k$ a tři ukázková $\gamma$ (šipky) s hodnotami, které z nich vyjdou. Vpravo je totéž jako **schodová distribuční funkce**: $\gamma$ vyneseme na svislou osu, doprava dojdeme k prvnímu schodu a dolů přečteme $k$.

In [ ]:
p_demo = [0.05, 0.1, 0.05, 0.6, 0.2]
cum_demo = [sum(p_demo[:k + 1]) for k in range(len(p_demo))]
barvy = plt.cm.tab10.colors
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1.3, 1]})

ax = axes[0]
zac = 0.0
for k, pk in enumerate(p_demo):
    ax.barh(0, pk, left=zac, color=barvy[k], edgecolor="k")
    popis = f"k = {k}\n{pk:g}"
    if pk >= 0.1:
        ax.text(zac + pk / 2, 0, popis, ha="center", va="center", fontsize=8)
    else:
        ax.text(zac + pk / 2, -0.5, popis, ha="center", va="top", fontsize=8)
    zac += pk
for u in (0.08, 0.46, 0.93):
    k = next(i for i, c in enumerate(cum_demo) if u < c)
    ax.annotate(f"γ = {u} → k = {k}", xy=(u, 0.4), xytext=(u, 1.0), ha="center", arrowprops=dict(arrowstyle="->"))
ax.set_ylim(-1.1, 1.4)
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_title("Interval [0, 1) rozdělený na úseky délek $p_k$")

ax = axes[1]
for k in range(len(p_demo)):
    ax.hlines(cum_demo[k], k - 0.5, k + 0.5, color=barvy[k], lw=3)
ax.step([-0.5] + list(range(len(p_demo))) + [len(p_demo) - 0.5], [0] + cum_demo + [1], where="mid", color="gray", lw=0.8)
u = 0.46
k = next(i for i, c in enumerate(cum_demo) if u < c)
ax.annotate("", xy=(k, u), xytext=(-0.5, u), arrowprops=dict(arrowstyle="->", color="r"))
ax.annotate("", xy=(k, 0), xytext=(k, u), arrowprops=dict(arrowstyle="->", color="r"))
ax.text(-0.45, u + 0.02, "γ = 0.46", color="r")
ax.set_xticks(range(len(p_demo)))
ax.set_xlabel("k")
ax.set_ylabel("$F(k)$")
ax.set_title("Distribuční funkce: $\\gamma$ → první $k$ s $\\gamma < F(k)$")
plt.tight_layout()
plt.show()

In [ ]:
def linear(p):
    '''Základní algoritmus: S = γ; odečítáme p_k, dokud není S < 0. Vrací (index, počet kroků).'''
    s = random.random()
    for k, pk in enumerate(p):
        s -= pk
        if s < 0:
            return k, k + 1
    return len(p) - 1, len(p)  # pojistka proti zaokrouhlení, kdyby bylo sum(p) o chlup menší než 1


p = [0.05, 0.1, 0.05, 0.6, 0.2]
hodnoty = [0, 1, 2, 3, 4]
presna_stredni = sum(v * q for v, q in zip(hodnoty, p))

n = 200_000
m, se = estimate(lambda: linear(p)[0], n)
check("E X (linear)", m, se, presna_stredni)

# četnosti ze simulace vs. zadané pravděpodobnosti
cetnosti = [0] * len(p)
for _ in range(n):
    cetnosti[linear(p)[0]] += 1

x = range(len(p))
plt.figure(figsize=(6.5, 3.8))
plt.bar([i - 0.2 for i in x], p, width=0.4, label="zadané $p_k$")
plt.bar([i + 0.2 for i in x], [c / n for c in cetnosti], width=0.4, label="ze simulace")
plt.xlabel("k")
plt.title("Základní algoritmus: četnosti odpovídají $p_k$")
plt.legend()
plt.show()

### Zvláštní případ: Bernoulliho veličina

Pro dvě hodnoty $\{0, 1\}$ s $P(X = 1) = q$ se algoritmus smrskne na jedno porovnání:

In [ ]:
def bernoulli(q):
    '''1 s pravděpodobností q, jinak 0.'''
    return 1 if random.random() < q else 0


m, se = estimate(lambda: bernoulli(0.3), n)
check("Bernoulli(0.3)", m, se, 0.3)

## 2. Kolik to stojí? Přeuspořádání

Počet kroků algoritmu je $k + 1$, když vyjde hodnota $k$. Střední počet kroků je tedy

$$E[\text{kroky}] = \sum_k (k+1)\,p_k.$$

Pořadí hodnot můžeme **libovolně zvolit** (je to jen naše značení), a proto je výhodné řadit je **sestupně podle pravděpodobnosti**: nejpravděpodobnější hodnoty se najdou hned na začátku. Výsledek (kterou hodnotu vrátíme) se nemění, jen musíme na konci převést index zpět na původní hodnotu.

Jak pořadí mění cenu, vidíme na malém příkladu: hodnoty jsou úseky jednotkového intervalu a algoritmus je prochází **zleva doprava**, dokud $\gamma$ nenajde svůj úsek. Číslo pod úsekem je počet kroků, když $\gamma$ padne dovnitř. Velké úseky vpředu se trefí hned, a proto je sestupné pořadí nejlevnější.

In [ ]:
p_ukazka = [0.05, 0.1, 0.05, 0.6, 0.2]
variany = {
    "vzestupně (nejhorší)": sorted(p_ukazka),
    "původní pořadí": p_ukazka,
    "sestupně (nejlepší)": sorted(p_ukazka, reverse=True),
}
fig, axes = plt.subplots(len(variany), 1, figsize=(10, 4.2), sharex=True)
for ax, (nazev, pp) in zip(axes, variany.items()):
    zac = 0.0
    for j, pk in enumerate(pp):
        ax.barh(0, pk, left=zac, color=plt.cm.Blues(0.25 + 0.15 * j), edgecolor="k")
        ax.text(zac + pk / 2, 0, f"{j + 1}", ha="center", va="center", fontsize=11, fontweight="bold")
        zac += pk
    kroky_ = sum((j + 1) * pk for j, pk in enumerate(pp))
    ax.set_yticks([])
    ax.set_ylabel(nazev, rotation=0, ha="right", va="center", fontsize=9)
    ax.set_title(f"průměrně {kroky_:.2f} kroku", fontsize=9, loc="right", pad=2)
axes[-1].set_xlim(0, 1)
axes[-1].set_xlabel("interval [0, 1), číslo v úseku = počet kroků, když $\\gamma$ padne sem")
plt.tight_layout()
plt.show()

In [ ]:
def make_reordered(p):
    '''Navštěvuje hodnoty od nejpravděpodobnější. Vrací (původní index, počet kroků).'''
    poradi = sorted(range(len(p)), key=lambda i: -p[i])
    p_serazene = [p[i] for i in poradi]

    def sample():
        k, kroky = linear(p_serazene)
        return poradi[k], kroky

    return sample


kroky_puvodni = sum((k + 1) * q for k, q in enumerate(p))
p_sestupne = sorted(p, reverse=True)
kroky_serazene = sum((k + 1) * q for k, q in enumerate(p_sestupne))
print(f"střední počet kroků, původní pořadí:   {kroky_puvodni:.2f}")
print(f"střední počet kroků, sestupné pořadí:  {kroky_serazene:.2f}")

m, se = estimate(lambda: linear(p)[1], n)
check("kroky, původní pořadí", m, se, kroky_puvodni)

reordered = make_reordered(p)
m, se = estimate(lambda: reordered()[1], n)
check("kroky, sestupné pořadí", m, se, kroky_serazene)

# výsledek se nezměnil, jen cena
m, se = estimate(lambda: reordered()[0], n)
check("E X (přeuspořádaně)", m, se, presna_stredni)

## 3. Půlení intervalu (bisekce)

Odečítání je lineární v počtu hodnot. Když hodnot je hodně (stovky, tisíce), vyplatí se předpočítat **kumulativní součty** $F_k = p_0 + \dots + p_k$ (rostoucí posloupnost) a hledat nejmenší $k$ s $\gamma < F_k$ **půlením intervalu**:

- předpočítání: $O(n)$, jednou
- každé losování: $O(\log n)$

Invariant: hledané $k$ vždy leží v $[\text{lo}, \text{hi}]$; porovnáním s prostředkem interval zmenšíme na polovinu.

Průběh na příkladu s $16$ hodnotami (pravděpodobnosti rostou s $k$): vlevo distribuční funkce s vyznačenými **porovnáními**, vpravo **zužující se interval** $[\text{lo}, \text{hi}]$ po jednotlivých krocích. Po každém porovnání se interval zmenší na polovinu, takže po $\lceil\log_2 16\rceil = 4$ krocích zbývá jedna hodnota.

In [ ]:
nb_ = 16
w16 = [k + 1 for k in range(nb_)]
p16 = [x / sum(w16) for x in w16]
cum16 = list(accumulate(p16))
g16 = 0.37

lo, hi = 0, nb_ - 1
kroky_b = []
while lo < hi:
    mid = (lo + hi) // 2
    vlevo = g16 < cum16[mid]
    kroky_b.append((lo, hi, mid, vlevo))
    if vlevo:
        hi = mid
    else:
        lo = mid + 1
vysledne_k = lo

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

ax = axes[0]
ax.step(range(nb_), cum16, where="mid", color="gray", lw=1)
ax.axhline(g16, color="r", ls="--")
ax.text(0, g16 + 0.02, f"γ = {g16}", color="r")
for i, (lo_, hi_, mid_, vlevo) in enumerate(kroky_b, 1):
    ax.plot([mid_], [cum16[mid_]], "o", color="tab:green" if vlevo else "tab:blue", ms=9)
    ax.text(mid_ + 0.25, cum16[mid_] - 0.05, f"{i}", fontsize=10)
ax.plot([vysledne_k], [cum16[vysledne_k]], "*", color="r", ms=16, label=f"výsledek k = {vysledne_k}")
ax.set_xlabel("k")
ax.set_ylabel("$F(k)$")
ax.set_title("Porovnání $\\gamma < F(\\mathrm{mid})$: zelená = ano (jdi doleva), modrá = ne")
ax.legend(loc="lower right")

ax = axes[1]
for i, (lo_, hi_, mid_, vlevo) in enumerate(kroky_b):
    ax.barh(-i, hi_ - lo_ + 1, left=lo_ - 0.5, color="tab:gray", alpha=0.3, edgecolor="k")
    ax.plot([mid_], [-i], "o", color="tab:green" if vlevo else "tab:blue")
    ax.text(nb_ - 0.2, -i, f"krok {i + 1}: [{lo_}, {hi_}], mid = {mid_}", va="center", fontsize=8)
ax.plot([vysledne_k], [-len(kroky_b)], "*", color="r", ms=16)
ax.text(nb_ - 0.2, -len(kroky_b), f"hotovo: k = {vysledne_k}", va="center", fontsize=8)
ax.set_xlim(-0.7, nb_ + 6.5)
ax.set_xticks(range(0, nb_, 2))
ax.set_yticks([])
ax.set_xlabel("k")
ax.set_title("Hledaný interval $[\\mathrm{lo}, \\mathrm{hi}]$ se po každém kroku zmenší na polovinu")
plt.tight_layout()
plt.show()

In [ ]:
def make_bisect(p):
    '''Jednou předpočítáme kumulativní součty, každé losování pak stojí O(log n).'''
    cum, soucet = [], 0.0
    for pk in p:  # knihovní ekvivalent: list(itertools.accumulate(p))
        soucet += pk
        cum.append(soucet)

    def sample():
        gamma = random.random()
        # knihovní ekvivalent: bisect.bisect_right(cum, gamma)
        lo, hi = 0, len(cum) - 1   # hi je i záchrana, kdyby cum[-1] < 1 kvůli zaokrouhlení
        while lo < hi:
            mid = (lo + hi) // 2
            if gamma < cum[mid]:
                hi = mid
            else:
                lo = mid + 1
        return lo

    return sample


def make_bisect_lib(p):
    '''Totéž ze standardní knihovny.'''
    cum = list(accumulate(p))
    return lambda: min(bisect_right(cum, random.random()), len(p) - 1)


m, se = estimate(make_bisect(p), n)
check("E X (bisect)", m, se, presna_stredni)
m, se = estimate(make_bisect_lib(p), n)
check("E X (bisect, knihovna)", m, se, presna_stredni)

## 4. Která metoda je kdy nejlepší?

Vyzkoušíme to na rozdělení s $1000$ hodnotami, $p_k \propto 1/(k+1)^2$ (málo hodnot má velkou pravděpodobnost, dlouhý chvost má zbytek). Porovnáme čtyři varianty:

1. lineární, hodnoty seřazeny **vzestupně** podle pravděpodobnosti (nejhorší pořadí),
2. lineární, hodnoty v **náhodném pořadí**: neutrální případ, když o pořadí vůbec nepřemýšlíme (v praxi přirozené pořadí hodnot, například $0, 1, 2, \dots$, bývá právě takhle „náhodné" vůči pravděpodobnostem; jestli vyjde lépe nebo hůř, záleží na tvaru rozdělení),
3. lineární, hodnoty seřazeny **sestupně** (nejlepší pořadí),
4. bisekce.

Jedna konkrétní náhodná permutace může mít štěstí nebo smůlu (záleží, kam padne nejpravděpodobnější hodnota), proto pro náhodné pořadí **zprůměrujeme přes sto permutací**. V průměru je každá hodnota na každé pozici stejně pravděpodobně, takže střední počet kroků vyjde $(n+1)/2$ bez ohledu na tvar rozdělení.

In [ ]:
nvals = 1000
w = [1 / (k + 1) ** 2 for k in range(nvals)]
soucet = sum(w)
p_desc = [x / soucet for x in w]          # sestupně: nejpravděpodobnější první
p_asc = p_desc[::-1]                      # vzestupně: nejhorší pořadí pro lineární hledání

draws = 20_000
vysledky = []

def zmer(nazev, sampler):
    t0 = time.perf_counter()
    kroky, _ = estimate(sampler, draws)
    t = (time.perf_counter() - t0) / draws
    vysledky.append((nazev, kroky, t))


zmer("lineární, vzestupně", lambda: linear(p_asc)[1])

# náhodné pořadí: 100 různých permutací po 200 losováních (jedna permutace může mít štěstí nebo smůlu)
kroky_perm = []
t0 = time.perf_counter()
for _ in range(100):
    p_rand = p_desc[:]
    random.shuffle(p_rand)
    kroky_perm.append(estimate(lambda: linear(p_rand)[1], draws // 100)[0])
t_rand = (time.perf_counter() - t0) / draws
vysledky.append(("lineární, náhodně", sum(kroky_perm) / len(kroky_perm), t_rand))

zmer("lineární, sestupně", lambda: linear(p_desc)[1])

bis = make_bisect(p_asc)
t0 = time.perf_counter()
for _ in range(draws):
    bis()
t = (time.perf_counter() - t0) / draws
vysledky.append(("bisekce", math.ceil(math.log2(nvals)), t))

print(f"{'metoda':22s} {'kroků na losování':>18s} {'čas na losování':>17s}")
for nazev, kroky, t in vysledky:
    print(f"{nazev:22s} {kroky:18.1f} {t * 1e6:14.1f} us")
print(f"\nnáhodné pořadí: teorie (n + 1) / 2 = {(nvals + 1) / 2:.1f}; "
      f"jednotlivé permutace daly od {min(kroky_perm):.0f} do {max(kroky_perm):.0f} kroků")

Totéž graficky (hodnoty z tabulky výše):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
nazvy = [v[0] for v in vysledky]
barvy_ = ["tab:red", "tab:olive", "tab:orange", "tab:green"]
b0 = axes[0].barh(nazvy, [v[1] for v in vysledky], color=barvy_)
axes[0].set_xscale("log")
axes[0].bar_label(b0, fmt="%.3g", padding=3, fontsize=8)
axes[0].margins(x=0.3)
axes[0].set_xlabel("průměrný počet kroků (log)")
axes[0].set_title("Počet kroků na losování")
b1 = axes[1].barh(nazvy, [v[2] * 1e6 for v in vysledky], color=barvy_)
axes[1].set_xscale("log")
axes[1].bar_label(b1, fmt="%.3g", padding=3, fontsize=8)
axes[1].margins(x=0.3)
axes[1].set_xlabel("čas na losování [µs] (log)")
axes[1].set_title("Čas na losování")
plt.tight_layout()
plt.show()

Vidíme, že:

- **Pořadí je klíčové** pro lineární algoritmus: vzestupné pořadí je mnohonásobně horší než sestupné.
- **Náhodné pořadí** je mezi nimi: v průměru $(n+1)/2$ kroků, takže jen asi dvakrát lepší než nejhorší případ a řádově horší než sestupné. Kdo o pořadí nepřemýšlí, nezíská z lineárního hledání nic.
- **Bisekce nezávisí na pořadí** a má zaručeně $\lceil\log_2 n\rceil$ kroků.
- Při velmi nerovnoměrném rozdělení může **lineární s dobrým pořadím** vyhrát i nad bisekcí (střední počet kroků je malý). U rovnoměrnějších pravděpodobností vyhrává bisekce.

Výběr tedy závisí na tvaru rozdělení: znáte-li ho a je výrazně nerovnoměrné, seřaďte a použijte lineární; jinak bisekce.

## 5. Poissonovo rozdělení

Poissonovo rozdělení $P(X = k) = \dfrac{\lambda^k e^{-\lambda}}{k!}$ má **nekonečně mnoho** hodnot, takže tabulku $p_k$ předem nesestavíme. Existuje ale elegantní algoritmus.

### Odvození

Počet událostí Poissonova procesu s intenzitou $1$ v intervalu $[0, \lambda]$ má rozdělení $\operatorname{Poisson}(\lambda)$. Mezery mezi událostmi jsou nezávislé exponenciální $E_i = -\ln \gamma_i$ (cvičení 4). Počet událostí v $[0, \lambda]$ je proto největší $k$, pro které $E_1 + \dots + E_k \le \lambda$. To lze přepsat:

$$E_1 + \dots + E_{k} \le \lambda
\iff -\ln(\gamma_1 \cdots \gamma_k) \le \lambda
\iff \gamma_1 \gamma_2 \cdots \gamma_k \ge e^{-\lambda}.$$

**Algoritmus:** násobíme rovnoměrná čísla $\gamma_1, \gamma_2, \dots$, dokud součin neklesne pod $e^{-\lambda}$. Jestliže jsme k tomu potřebovali $K$ čísel, je výsledek $K - 1$ (poslední číslo „přestřelilo", takže se do $[0, \lambda]$ vešlo o jedno méně).

Ilustrace: každý řádek je jeden pokus. Body jsou časy událostí $E_1,\ E_1 + E_2,\ \dots$ (součty exponenciálních mezer) na časové ose. Hodnota Poissonovy veličiny je počet bodů, které se vejdou do $[0, \lambda]$ (zde $\lambda = 4$, svislá čára).

In [ ]:
lam_demo = 4.0
fig, ax = plt.subplots(figsize=(10, 3.6))
for radek in range(6):
    t, body = 0.0, []
    while t <= lam_demo + 3:
        t += -math.log(1.0 - random.random())   # mezera Exp(1)
        body.append(t)
    body = [b for b in body if b <= lam_demo + 3]
    uvnitr = [b for b in body if b <= lam_demo]
    ax.scatter(body, [radek] * len(body), c=["tab:green" if b <= lam_demo else "lightgray" for b in body], s=30)
    ax.text(lam_demo + 3.3, radek, f"X = {len(uvnitr)}", va="center")
ax.axvline(lam_demo, color="k", ls="--")
ax.set_xlim(0, lam_demo + 4.5)
ax.set_yticks([])
ax.set_xlabel("čas")
ax.set_title(r"Poissonův proces: počet událostí v $[0, \lambda]$ je Poisson($\lambda$)")
plt.tight_layout()
plt.show()

### Pozor na chybu o jedničku

Následující kód je **správně vypadající**, ale vrací $K$ místo $K - 1$:

In [ ]:
def poisson_s_chybou(lam):
    q = math.exp(-lam)
    n = 1
    s = random.random()
    while s > q:
        s *= random.random()
        n += 1
    return n            # BUG: počet použitých čísel K, správně je K - 1


def poisson(lam):
    '''Poissonovo rozdělení násobením rovnoměrných čísel.'''
    q = math.exp(-lam)
    n = 1
    s = random.random()
    while s > q:
        s *= random.random()
        n += 1
    return n - 1


lam = 4.0
n = 200_000

for nazev, gen in (("s chybou", poisson_s_chybou), ("opraveno", poisson)):
    xs = [gen(lam) for _ in range(n)]
    print(f"{nazev:9s}: průměr {sum(xs) / n:.3f} (má být {lam}),  minimum {min(xs)}")

S chybou je střední hodnota $\lambda + 1$ a **nikdy nevyjde $0$**. Vidíte to na prvním pohledu na histogram, ale hlavně to **hned odhalí test** se střední hodnotou, který máme v celém kurzu zabudovaný:

In [ ]:
for nazev, gen in (("s chybou", poisson_s_chybou), ("opraveno", poisson)):
    try:
        m, se = estimate(lambda: gen(lam), n)
        check(f"E X ({nazev})", m, se, lam)
    except AssertionError:
        print(f"E X ({nazev}): test SELHAL, odhad {m:.4f} je daleko od {lam}")

> **Poučení:** každý generátor ověřte proti něčemu, co znáte přesně: střední hodnotou, rozptylem, pravděpodobností jedné hodnoty. Chyba o jedničku je mezi nejběžnějšími a prohlédnutí histogramu ji často neodhalí.

### Porovnání s přesnými pravděpodobnostmi

In [ ]:
xs = [poisson(lam) for _ in range(n)]
m = sum(xs) / n
v = sum((x - m) ** 2 for x in xs) / (n - 1)
print(f"střední hodnota {m:.3f}, rozptyl {v:.3f}  (obojí má být {lam}, u Poissonova rozdělení jsou stejné)")

kmax = 15
cetnosti = [xs.count(k) / n for k in range(kmax)]
presne = [lam ** k * math.exp(-lam) / math.factorial(k) for k in range(kmax)]

plt.figure(figsize=(7, 3.8))
plt.bar(range(kmax), cetnosti, alpha=0.6, label="simulace")
plt.plot(range(kmax), presne, "ro", label="přesně")
plt.xlabel("k")
plt.title(f"Poisson({lam:g}): simulace vs. přesné pravděpodobnosti")
plt.legend()
plt.show()

### Cena

Střední počet použitých rovnoměrných čísel je $\lambda + 1$ (součin klesá o faktor $e^{-1}$ na jedno číslo), takže generování je **lineární v $\lambda$**. Pro velké $\lambda$ (stovky) je algoritmus pomalý a $e^{-\lambda}$ navíc **podteče**: od $\lambda \gtrsim 745$ je v počítači přesně $0$. Součinová verze pak vrací nesmysl (hodnoty kolem 700–800 bez ohledu na $\lambda$) a inverzní verze s rekurencí níže se **zacyklí** (už při $\lambda = 800$). Pro velká $\lambda$ se používají jiné metody (například přibližné rozdělení normální nebo zamítací metoda).

### Alternativa: základní algoritmus s rekurencí

Poissonovo rozdělení lze generovat i základním algoritmem z bodu 1, jen hodnoty $p_k$ nepočítáme předem, ale **za pochodu** z rekurence $p_{k+1} = p_k\,\lambda/(k+1)$:

In [ ]:
def poisson_inverzni(lam):
    '''Základní algoritmus s rekurencí p_{k+1} = p_k * lam / (k + 1).'''
    u = random.random()
    k = 0
    pk = math.exp(-lam)
    soucet = pk
    while u >= soucet:
        k += 1
        pk *= lam / k
        soucet += pk
    return k


m, se = estimate(lambda: poisson_inverzni(lam), n)
check("E X (poisson, inverzní)", m, se, lam)

t0 = time.perf_counter()
estimate(lambda: poisson(lam), n)
t1 = time.perf_counter()
estimate(lambda: poisson_inverzni(lam), n)
t2 = time.perf_counter()
print(f"čas na losování: součin čísel {(t1 - t0) / n * 1e6:.2f} us,  inverzní s rekurencí {(t2 - t1) / n * 1e6:.2f} us")

Obě metody dávají totéž rozdělení. Inverzní varianta spotřebuje **jedno** rovnoměrné číslo na losování (součinová potřebuje v průměru $\lambda + 1$), což se hodí, když je generátor drahý nebo když potřebujeme opakovatelnost.

## 6. Geometrické rozdělení

Počet „nezdarů" před prvním „úspěchem" v nezávislých pokusech s pravděpodobností úspěchu $p$ má $P(G = k) = (1-p)^k p$. Distribuční funkce $P(G \le k) = 1 - (1-p)^{k+1}$ jde invertovat **uzavřeným vzorcem**, takže nepotřebujeme žádný cyklus:

$$G = \left\lfloor \frac{\ln \gamma}{\ln(1-p)} \right\rfloor.$$

> Tenhle vzorec použijeme ve **cvičení 9**: v modelu pojišťovny jím přeskakujeme dny bez požadavku, takže místo $N$ losování uděláme jen tolik, kolik je požadavků.

(Právě tento vzorec jsme použili ve cvičení 2 na přeskakování intervalů bez požadavku.)

In [ ]:
def geometricke(p):
    '''Počet nezdarů před prvním úspěchem; 1 - γ je také rovnoměrné a nikdy 0.'''
    return int(math.log(1.0 - random.random()) / math.log(1.0 - p))


pp = 0.2
m, se = estimate(lambda: geometricke(pp), n)
check("E G", m, se, (1 - pp) / pp)

m, se = estimate(lambda: float(geometricke(pp) == 3), n)
check("P(G = 3)", m, se, (1 - pp) ** 3 * pp)

## 7. Alternativa: NumPy

NumPy má hotové generátory pro všechna tato rozdělení a umí generovat celé pole najednou.

In [ ]:
try:
    import numpy as np

    rng = np.random.default_rng(1)
    N = 1_000_000

    t0 = time.perf_counter()
    x1 = rng.choice(len(p), size=N, p=p)                 # diskrétní rozdělení z cvičení 1
    x2 = rng.poisson(lam, size=N)
    x3 = rng.geometric(pp, size=N) - 1                   # NumPy počítá pokusy, ne nezdary
    t = time.perf_counter() - t0

    print(f"E X (choice) = {x1.mean():.4f}   (přesně {presna_stredni:.4f})")
    print(f"E X (poisson) = {x2.mean():.4f}, rozptyl = {x2.var():.4f}   (obojí {lam})")
    print(f"E G (geometric) = {x3.mean():.4f}   (přesně {(1 - pp) / pp:.4f})")
    print(f"{3 * N:,} vzorků za {t:.2f} s".replace(",", " "))
except ModuleNotFoundError:
    print("NumPy není nainstalované, spusťte:  uv sync")

Pozor na rozdíl v konvencích: `numpy.random.Generator.geometric` počítá **pokusy včetně úspěšného** (hodnoty $1, 2, \dots$), my počítáme **nezdary** ($0, 1, \dots$). Proto odečítáme jedničku. Takové rozdíly mezi knihovnami jsou častým zdrojem chyb.

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Základní algoritmus | $S = \gamma$, odečítej $p_k$, dokud $S < 0$; vrať $k$ (inverzní transformace) |
| Cena | $E[\text{kroky}] = \sum (k+1)p_k$; řaďte hodnoty **sestupně** podle $p_k$ |
| Bisekce | předpočítat kumulativní součty $O(n)$, pak $O(\log n)$ na losování; nezávisí na pořadí |
| Volba metody | silně nerovnoměrné $p_k$: seřazená lineární; jinak bisekce |
| Poisson | násob $\gamma_i$, dokud součin $< e^{-\lambda}$; výsledek je počet čísel **minus 1**; cena $\sim\lambda$ |
| Geometrické | $\lfloor \ln \gamma / \ln(1-p)\rfloor$, uzavřený vzorec bez cyklu |
| Testování | každý generátor ověřit střední hodnotou a rozptylem; chyba o jedničku se hledá hůř, než se zdá |

### Zkuste si

1. Pomocí `linear` a `make_bisect` vygenerujte **součet dvou kostek** (hodnoty $2, \dots, 12$ s pravděpodobnostmi $1/36, 2/36, \dots, 6/36, \dots, 1/36$). Ověřte střední hodnotu $7$ funkcí `check`.
2. Zjistěte, od jakého $\lambda$ je `poisson_inverzni` pomalejší než `poisson`, nebo obráceně. Je to $\lambda$ blízko $1$, nebo $10$?
3. Vygenerujte **binomické** rozdělení $\operatorname{Bi}(n, p)$ třemi způsoby: součtem $n$ Bernoulliho veličin, základním algoritmem s rekurencí $p_{k+1} = p_k\,\frac{n-k}{k+1}\,\frac{p}{1-p}$ a pomocí `rng.binomial`. Kolik rovnoměrných čísel která metoda spotřebuje?